# Notebook 4 - Memory and Multi-step Tasks

## Why this notebook exists

Each run starts cold. That's fine for one question, but a research assistant should remember what you told it an hour ago, or even three steps ago; if it cannot, it is not an assistant.

Memory is what lets an agent carry context across steps and across sessions. By the end of this notebook, you will be able to explain the difference between short-term and long-term memory, identify when each is appropriate, and build a multi-step task where two tool calls are connected through memory.

## What we'll build
1. **Short-term memory**
2. **Long-term memory**
3. **Semantic memory**
4. **A multi-step task** that only works if the agent remembers an earlier step.

## Prerequisites
- A Google account (for Google Colab, no installation).
- Notebooks 1 and 2 completed, or familiarity with `call_llm` and `run_agent`.
- Default (mock mode): NO key needed. The notebook runs offline with predefined responses.
- Live mode: requires an `ANTHROPIC_API_KEY`. Set `PROVIDER = "anthropic"` and rerun from the top.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/04_memory_multistep.ipynb)

## Step 1 - Setup

**Note:** This notebook supports `anthropic` and `mock` only.

*Provider-agnostic* means `call_llm()` is the single call point. Set PROVIDER once; nothing else changes.

| Function | Role |
|------|-----|
| `mock_reset(script)` | Loads scripted responses. Call before each demo cell. |
| `call_llm(...)` | The only function you call directly |

**How to add your Anthropic API key in Colab:** 

1. Click the key icon in the left sidebar --> Add new secret
2. Name it exactly `ANTHROPIC_API_KEY`
3. Turn on `Notebook access`
4. Set `PROVIDER = "anthropic"` and re-run from the top

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothing else in the notebook changes. An agent is a pattern, not a vendor.
#  In this notebook, we install embedding and numeric packages here

# In Google Colab this cell installs the packages. Locally, run once.

%pip install -q anthropic numpy

import os
import json

PROVIDER = "mock"  # @param ["mock", "anthropic"]

MODEL = "claude-haiku-4-5-20251001"  # cheapest current Claude model

if PROVIDER == "anthropic":
    try:
        from google.colab import userdata
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    except Exception:
        pass


# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}

_MOCK_SCRIPT = []
def mock_reset(script):
    """Load a scripted response sequence for mock mode.
    script: list of tuples.
    ("final", "text string") -- returns a text reply.
    ("tool", {"name": "...", "args": {...}}) -- returns a tool call.
    Call this before every demo cell to reset the script.
    """
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''
    The one function every cell calls. In mock mode replays the queued script; in live mode calls the Anthropic API.
    messages: List of dicts with role and content keys.
    tools: List of tool-description dicts (same format as tools_spec).
    system: None = no system prompt. A string sets a persistent instruction that shapes all replies without being part of the conversation. For example: "You are a concise research assistant."
    max_tokens:  Maximum reply length (default 1024).
    temperature: 0 = deterministic output; higher = more varied.
    Returns: Dict with keys: text (str), tool_calls (list), stop_reason (str).
    '''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw = dict(model=MODEL, max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    raise ValueError(f"PROVIDER must be 'anthropic' or 'mock'; got {PROVIDER!r}")

print(f"Setup ready. PROVIDER={PROVIDER!r}. Commercial provider: Anthropic only.")

## Carried from Notebooks 1, 2, and 3

The agent in this notebook uses the same control structure from Notebook 1 — the model decides what to do next. The new element is memory: the agent now *remembers* what it did before, which is what makes multi-step tasks possible.

`run_agent` comes from Notebook 2. The embedding retrieval idea in Step 4 (optional) comes from Notebook 3.

In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.

    Arguments:   
       user_task: Plain string — the task you give the agent.
       tools_spec: List of tool-description dicts (the model reads these to know what tools exist).
       tool_fns: Dict mapping tool name to Python function — used to run the tool when called.
       system: None or a string instruction. Shapes how the agent reasons without being part of the task. Example: "Be concise."
       max_steps: Safety ceiling on tool calls (default 6). Returns "Stopped: hit max_steps." if reached. Increase for longer tasks.
       verbose: True prints each tool call and final step live. False for silent runs.
    '''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

## Step 2 - Short-term memory

The simplest memory is just *keeping the transcript* and sending it back every turn. Watch what happens with and without it.

**What `chat_with_memory(history, user_msg)` does:**
- history — a list of message dicts, the growing conversation transcript. Pass [] to start fresh (no memory).
- user_msg — the new message to send.
- It appends the new user message to history, calls call_llm, appends the assistant reply, then returns the updated history and the reply text.

The *memory* here is just the list. Passing hist back into the next call is what makes the model remember — the whole conversation is literally resent to the model each turn. 
The *WITHOUT memory* case shows what happens when you pass [] each time instead.

> **Mock mode:** the responses below are pre-loaded with `mock_reset(...)`. In mock mode, predefined text is returned — a stand-in for the model, not the model. The memory mechanics (appending to `hist`, passing it back each turn) are real and unchanged.

In [ ]:
SHORT_TERM_MSG = "My project is MS-CC-AI-Enablement Series."  # @param {type:"string"}

In [ ]:
def chat_with_memory(history, user_msg):
    """Send one message to the model, keeping conversation history.
      history:  List of message dicts (the growing transcript). Pass [] to start fresh.
      user_msg: The new user message as a plain string.
      Returns:  (updated_history, reply_text) — pass the updated history into the next call to give the model memory of earlier turns.

      To customize: try different system prompts by passing system= to call_llm, or limit history length by slicing: history[-10:] before passing it back.
    """
    history = history + [{"role":"user","content":user_msg}]
    reply = call_llm(history)
    history = history + [{"role":"assistant","content":reply["text"]}]
    return history, reply["text"]

# With memory: the agent has the earlier turn in context.
mock_reset([("final", f"Got it, I will remember that: {SHORT_TERM_MSG}"),
            ("final", f"Earlier you told me: {SHORT_TERM_MSG}")])
hist=[]
hist,_ = chat_with_memory(hist, SHORT_TERM_MSG + " Remember that.")
hist, ans = chat_with_memory(hist, "What did I just tell you?")
print("WITH memory -->", ans)

# Without memory: each call starts fresh, so it cannot know.
mock_reset([("final","I don't have any earlier context. Could you remind me?")])
_, ans2 = chat_with_memory([], "What did I just tell you?")
print("WITHOUT memory -->", ans2)

### What's the cost of a buffer?

For longer conversations, resending the full transcript every turn becomes expensive and eventually hits the model's context limit. We need long-term memory, so that we can reduce the cost and prevent the context window from overflowing, and we can keep the useful facts without dragging the entire transcript along. 

In [ ]:
# Each turn re-sends the full conversation history, so input tokens compound — not add.
# Turn 1: 120 tokens. Turn 2: 120+140=260. Turn 3: 420. Turn 4: 570.
# Long sessions get expensive fast. Long-term memory is the fix: store facts, not transcripts.
print("tokens billed per turn (cumulative):", [sum([120, 140, 160, 150][:i+1]) for i in range(4)])

## Step 3 - Long-term memory

Long-term memory stores facts in a persistent location that survives between sessions. We implement this as a Python dictionary saved to a JSON file - something you can open, read, and edit by hand.

**What the Memory class does:**

| Method | What it does |
|---|----|
| `Memory(path)` | Loads the JSON file at path if it exists; otherwise starts empty |
| `remember(key, value)` | Stores the value, immediately writes the whole dict to disk |
| `recall(key, default)` | Returns the stored value, or default if the key doesn't exist |
| `dump()` | Returns the full dict - useful for inspecting what the agent has stored |


In [ ]:
class Memory:
    """
    Persistent key-value store backed by a JSON file.    
    Each call to remember() writes immediately to disk, so facts survive between sessions and across notebook restarts. The file is human-readable and can be opened, edited, or deleted by hand.
    path: file path for the JSON store (default "agent_memory.json").
    """
    def __init__(self, path="agent_memory.json"):
        self.path = path
        if os.path.exists(path):
            with open(path) as f:
                self.data = json.load(f)
        else:                           
            self.data = {}
    def remember(self, key, value):
      self.data[key] = value
      with open(self.path, "w") as f:
          json.dump(self.data, f, indent=2)
      return value
    def recall(self, key, default=None):
        return self.data.get(key, default)
    def dump(self):
        return dict(self.data)

mem = Memory("agent_memory.json")
mem.remember("project", "Prompt Tree framework dissertation")
mem.remember("nsf_deadline", "2026-11-03")

# Simulate a brand-new session: make a fresh object from the same file.
new_session = Memory("agent_memory.json")
print("Recalled in a new session:", new_session.recall("project"), "| deadline:", new_session.recall("nsf_deadline"))
print("Everything stored:", new_session.dump())

> **Open `agent_memory.json` now** (in the file browser on the left). You will see the two keys you just stored — `project` and `nsf_deadline` — as plain JSON. This is the agent's long-term memory. It survives a notebook restart because it is a file on disk, not a Python variable.

## Step 4 - Semantic memory *(optional)*

> **Skip this step if you're short on time.** Semantic memory applies the same embedding idea from Notebook 3 to the agent's own stored memories — same technique, new target. It adds no new concept. Come back to it as self-study.

Key-value memory only works if you ask for the *exact* key. Real questions don't work like that. If you stored "the NSF proposal is due Nov 3" and later ask "when's the submission due?", a dict lookup fails.

Semantic memory embeds each stored memory and the query, then returns the closest match by cosine similarity — the same retrieval from Notebook 3, now pointed at the agent's own notes.

**What SemanticMemory does:**

- `sm.add(text)` — stores the text and re-embeds all stored memories
- `sm.recall(query, k=1)` — returns the k closest memories with their scores (0 = unrelated, 1.0 = identical)

In [ ]:
import re
import math
import numpy as np

# Common words excluded from TF-IDF scoring (stopwords)
_STOP = set("the a an is are of to in on for my your it and with be as at by from that this".split())

def _semantic_embedder(memories):
    """
    Returns an embedding function and its label for the given list of memory strings.
    Tries sentence-transformers first; falls back to TF-IDF if not installed.

    memories: list of strings to build the vocabulary from (for TF-IDF fallback).
    Returns: callable(texts) -> (numpy array of embeddings, label string)
    """
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("all-MiniLM-L6-v2")
        return lambda texts: (np.array(model.encode(list(texts))), "sentence-transformers")
    except Exception: pass
    # internal: TF-IDF fallback — not part of the lesson.
    vocab = {}
    for t in memories:
        for w in set(re.findall(r"[a-z]+", t.lower())):
            if w not in _STOP: vocab[w] = vocab.get(w, 0) + 1
    N = max(len(memories), 1)
    idf = {w: math.log((N+1)/(df+1))+1 for w, df in vocab.items()}
    index = {w: i for i, w in enumerate(sorted(idf))}
    def emb(texts):
        M = np.zeros((len(texts), len(index)))
        for r, t in enumerate(texts):
            for w in re.findall(r"[a-z]+", t.lower()):
                if w in index: M[r, index[w]] += idf[w]
        return M, "tfidf-fallback"
    return emb

class SemanticMemory:
    """
    Memory store that retrieves entries by semantic similarity rather than exact key.

    Use sm.add(text) to store a fact.
    Use sm.recall(query, k=1) to retrieve the k most similar facts.

    Uses sentence-transformers for real semantic search if installed;
    falls back to TF-IDF (keyword overlap only) with no extra setup required.
    The embedder label (sm.kind) tells you which is active.
    """
    def __init__(self):
        self.memories = []
        self._vecs = None
        self.kind = "(none)"
    def add(self, text):
        self.memories.append(text)
        emb = _semantic_embedder(self.memories)
        self._vecs, self.kind = emb(self.memories)
        self._emb = emb
    def recall(self, query, k=1):
        if not self.memories:
            return []
        qv, _ = self._emb([query])
        qv = qv[0]
        sims = [float(qv @ v / ((np.linalg.norm(qv) * np.linalg.norm(v)) + 1e-9)) for v in self._vecs]
        order = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)[:k]
        return [(self.memories[i], round(sims[i], 3)) for i in order]

sm = SemanticMemory()
for fact in ["The NSF AI proposal is due November 3, 2026.",
             "My co-PI is Dr. Stark in the Education department.",
             "IRB requires human review before AI output affects a participant."]:
    sm.add(fact)
print("embedder in use:", sm.kind, "\n")

# Case 1: shares distinctive words with a memory -> works with ANY embedder.
print("Q1 (shared words) 'When is the NSF proposal due?' -->", sm.recall("When is the NSF proposal due?"))
# Case 2: pure paraphrase, no shared words -> needs REAL embeddings.
print("Q2 (paraphrase)  'Who is my collaborator?' -->", sm.recall("Who is my collaborator?"))

## Step 5 - A multi-step task that needs memory

The two tools below are designed to depend on each other through memory:

| Tool | What it does | Memory role |
|----|-------|------|
| `lookup_deadline(program)` | Looks up a deadline and writes it to `task_mem` | Producer — writes *last_deadline* |
| `days_until_remembered()` | Computes days from today to the stored deadline | Consumer — reads *last_deadline* |

`days_until_remembered` takes no arguments. It has no way to receive the deadline except via memory. This means the agent must call `lookup_deadline` first. If it skips that step, the second tool finds nothing in memory and returns an error.

This is the key takeaway: **memory** is what connects two otherwise independent tool calls into a genuine multi-step workflow.

After running the demo below, open `task_memory.json` in the file browser. You will see `last_deadline` stored as a plain JSON key. That file is the only link between the two tool calls — not a variable, not the loop, just a file on disk.

> **Mock mode:** the tool-call sequence and final answer below are pre-loaded with `mock_reset(...)`. In mock mode, the model's tool selections are scripted. The tool functions themselves — `lookup_deadline` and `days_until_remembered` — run normally and write real values to `task_memory.json`.

In [ ]:
MULTI_STEP_TASK = "Look up the NSF AI deadline, then tell me how many days I have left."  # @param {type:"string"}

In [ ]:
from datetime import date

task_mem = Memory("task_memory.json")

def lookup_deadline(program):
    # Look up a program's deadline and store it in memory.
    # Normalize input: "NSF AI" -> "nsf-ai" so live model calls match the key.
    deadlines = {"nsf-ai": "2026-11-03"}
    d = deadlines.get(program.lower().replace(" ", "-"), "unknown")
    task_mem.remember("last_deadline", d)      # <-- writes to memory
    return d

def days_until_remembered():
    # Compute days from today until the deadline stored in memory. Requires lookup_deadline to have run first
    d=task_mem.recall("last_deadline")
    if not d or d=="unknown": return "No deadline in memory; look one up first."
    return str((date.fromisoformat(d)-date.today()).days)

tools_spec=[
 {"name":"lookup_deadline","description":"Look up a program's deadline and store it in memory.",
  "parameters":{"type":"object","properties":{"program":{"type":"string"}},"required":["program"]}},
 {"name":"days_until_remembered","description":"Days from today until the deadline stored in memory.",
  "parameters":{"type":"object","properties":{},"required":[]}},
]
tool_fns={"lookup_deadline":lookup_deadline,"days_until_remembered":days_until_remembered}

mock_reset([
    ("tool",{"name":"lookup_deadline","args":{"program":"nsf-ai"}}),
    ("tool",{"name":"days_until_remembered","args":{}}),
    ("final","I looked up the NSF AI deadline (2026-11-03), stored it in memory, then calculated the days remaining from today."),
])
ans, _ = run_agent(MULTI_STEP_TASK, tools_spec, tool_fns)
print("\nAnswer:", ans)

## Step 6 - Human oversight: checkpoints, stopping conditions, and guardrails

Three mechanisms that keep a human in the loop when an agent runs tasks that affect real systems.

| Mechanism | What it does |
|---|---|
| **Checkpoint** | Pauses the loop at a specific step and shows what the agent is about to do next. Replace the `print` with `input()` to require approval before continuing. |
| **Stopping condition** | Halts the loop when a result falls outside acceptable bounds. `max_steps` is one. You can add others: deadline too far out, cost too high, quality too low. |
| **Guardrail** | Blocks specific actions before they execute — regardless of what the model decided. Here: certain memory keys cannot be written, ever. |

In [ ]:
# In mock mode: tool calls and final answer are scripted below.
mock_reset([
    ("tool",  {"name": "lookup_deadline",       "args": {"program": "nsf-ai"}}),
    ("tool",  {"name": "days_until_remembered", "args": {}}),
    ("final", "NSF AI deadline is 2026-11-03. Checkpoint fired at step 2 before the second tool call."),
])

# 1. CHECKPOINT — show what the agent is about to do before it does it.
#    In production: replace the print with input() to require human approval.
def run_agent_with_checkpoint(user_task, tools_spec, tool_fns, system=None,
                               max_steps=6, checkpoint_at=2):
    messages = [{"role": "user", "content": user_task}]
    for step in range(1, max_steps + 1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:
            checkpoint_tc = reply["tool_calls"][0]
            if step == checkpoint_at:
                print(f"\n  CHECKPOINT step {step}: agent wants `{checkpoint_tc['name']}` <- {checkpoint_tc['args']}")
                print("  (In production: pause here and ask a human before continuing.)\n")
            messages.append({"role": "assistant", "content": reply["text"],
                             "tool_calls": reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role": "tool", "tool_call_id": tc["id"],
                                 "name": tc["name"], "content": result})
        else:
            print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

ans, _ = run_agent_with_checkpoint(MULTI_STEP_TASK, tools_spec, tool_fns, checkpoint_at=2)
print("\nAnswer:", ans)

# 2. STOPPING CONDITION — halt when a result falls outside acceptable bounds.
#    max_steps is one stopping condition; here is a second: reject if deadline > 365 days out.
def is_within_planning_window(days_str, limit=365):
    """Return True only if the deadline is within the planning window."""
    try:    return 0 <= int(days_str) <= limit
    except: return False

# 3. GUARDRAIL — block writes to sensitive memory keys, regardless of what the model decides.
PROTECTED_KEYS = {"participant_id", "irb_approval"}

def guarded_remember(key, value, memory):
    if key in PROTECTED_KEYS:
        return f"BLOCKED: '{key}' is a protected key. Do not store sensitive identifiers."
    return memory.remember(key, value)

print("\nGuardrail — blocked key:")
print(" ", guarded_remember("participant_id", "P001", task_mem))
print("Guardrail — allowed key:")
print(" ", guarded_remember("last_deadline", "2026-11-03", task_mem))

## Step 7 - Exercise

1. Run the multi-step task above. It works because `lookup_deadline` writes to memory and `days_until_remembered` reads it.
2. Now **break the link**: comment out the `task_mem.remember(...)` line inside `lookup_deadline`, restart, and run the task again.
3. Watch the second tool fail to find anything.

**The point:** the two steps aren't connected by the loop — they're connected by *memory*. Remove memory and the "multi-step" agent falls back to two unrelated actions. Write down, in one sentence, what memory was actually doing here.

In [ ]:
# Simulate the break: clear memory between the two steps.
task_mem.data.pop("last_deadline", None)
with open("task_memory.json", "w") as f:
    json.dump(task_mem.data, f)
print("With memory broken, step 2 returns:")
print(" ", days_until_remembered())

## Reflection - and one hard line on memory

- **Cost**: memory keeps context cheap by storing facts instead of whole transcripts. Where would that help your longest interactions?
- **Staleness**: a remembered "deadline" can go out of date. How would you expire or refresh memories?
- **Privacy (the hard line)**: memory means the agent *keeps* things. Never persist identifiable human-subjects data, financial records, or access-restricted material in an agent's memory unless the vendor and protocol explicitly allow it (see datasets/irb_policy_ai.md). If you can't say where a memory is stored and who can read it, don't store it.